### Purpose
Cohort retention and fraud exposure by customer segment.

**Data:** `marts.fct_transactions` + dims (Postgres). Requires `make docker-up` + `dbt build`.

**Charts:** `notebooks/charts/customer_segments/`

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR))

from notebooks.utils.data_loader import get_merged_data, print_data_sanity
from notebooks.utils.style import C, INR_CR, apply_theme, style_axes
from notebooks.utils.visualization import minimal_heatmap, save_chart, new_fig

apply_theme()
CHART_SUBDIR = "customer_segments"

df, data_source = get_merged_data()
df["fraud_amount"] = df["amount"].where(df["is_fraud"], 0)
print_data_sanity(df, data_source)

### Chart - Cohort retention

In [ ]:
df["cohort"] = df.groupby("user_id")["transaction_ts"].transform("min").dt.to_period("M")
df["activity"] = df["transaction_ts"].dt.to_period("M")
df["offset"] = (df["activity"] - df["cohort"]).apply(lambda x: x.n)
coh = df[df["offset"].between(0, 5)]
sizes = coh.groupby("cohort")["user_id"].nunique()
active_users = coh.groupby(["cohort", "offset"])["user_id"].nunique().unstack(fill_value=0)
ret = active_users.div(sizes, axis=0).mul(100).round(0)
ret.index = ret.index.astype(str)

fig, ax = new_fig(9, 4.5)
minimal_heatmap(ret, ax, cbar_label="Retention %")
ax.set_title("User cohort retention (% active)", loc="left", fontweight="600")
ax.set_xlabel("Months since first txn")
ax.set_ylabel("Cohort")
save_chart(fig, "cohort_retention_matrix", CHART_SUBDIR)

`Sharp drop at month 3 is synthetic-user behaviour — in production, month-1 cliff is the key growth signal.`

### Chart - Fraud loss by customer segment

In [ ]:
seg = df.groupby("user_id").agg(fraud_txns=("is_fraud", "sum"), fraud_loss=("fraud_amount", "sum")).reset_index()
seg["segment"] = pd.qcut(seg["fraud_txns"], q=3, labels=["Low repeat", "Medium repeat", "High repeat"])
stats = (
    seg.groupby("segment", observed=True)
    .agg(users=("user_id", "count"), fraud_txns=("fraud_txns", "mean"), fraud_loss=("fraud_loss", "sum"))
    .reindex(["Low repeat", "Medium repeat", "High repeat"])
)
seg_colors = [C["success"], C["warning"], C["danger"]]

fig, ax = new_fig(8, 4.5)
bars = ax.bar(stats.index.astype(str), stats["fraud_loss"].to_numpy(), color=seg_colors, width=0.55, alpha=0.9)
ax.set_ylabel("Fraud loss (INR)")
ax.yaxis.set_major_formatter(INR_CR)
ax.set_title("Fraud loss by repeat-fraud intensity (user tertiles)", loc="left", fontweight="600")
save_chart(fig, "segment_users_fraud_loss_share", CHART_SUBDIR)

`Tertiles split users by fraud-txn count (~12 / 15 / 16+ avg). High-repeat users drive the largest ₹ loss despite similar headcount.`

### Segment summary

In [ ]:
total_loss = stats["fraud_loss"].sum()
for label, row in stats.iterrows():
    share = row["fraud_loss"] / total_loss * 100 if total_loss else 0
    print(f"{str(label):14} | {int(row['users']):5,} users | ₹{row['fraud_loss']:,.0f} ({share:.1f}% of fraud loss)")